**Eksport:** `EXPORT_DESTINATION = "local"` – pobierz bezpośrednio do `data/gee_rasters` (bez Drive/GCS). Alternatywy: `gcs`, `drive`.

# GEE Hex Terrain: Land Cover & Elevation for H3 Hexes

**Dwa etapy (bez zapytań do GEE przy agregacji):**
1. **GEE Export (jednorazowo)** – eksport rastrów do S3 (GEE→GCS→S3)
2. **Lokalne przetwarzanie** – rasterstats, odczyt z S3, zapis do S3

**Dane:** Copernicus 100m land cover, SRTM 30m elevation + slope

**S3:** `gold/gee_hex_terrain/country=ES/snapshot=2019/h3_resolution=6/`

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
# Repo root – działa niezależnie od lokalizacji notebooka (src/ vs src/gee/)
from pathlib import Path
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

# Land cover: Copernicus 100m (years 2015–2019)
LANDCOVER_YEAR = 2019
LANDCOVER_SCALE = 200

# Elevation + slope: SRTM (resample do 100m – szybszy eksport, ~100MB zamiast ~8GB)
ELEVATION_SCALE = 200
INCLUDE_SLOPE = True

# H3 resolutions to process (6=~36km², 7=~5km², 8=~0.7km², 9=~0.1km²)
# Spain: res6 ~14k, res7 ~96k, res8 ~675k, res9 ~5M
H3_RESOLUTIONS = [6, 7, 8, 9]

# Region
COUNTRY = "ES"  # ISO country code for partition
SNAPSHOT = str(LANDCOVER_YEAR)  # partition: land cover year (or date e.g. 2024-02-15)
USE_SPAIN_BOUNDARY = True  # from FAO GAUL level0
FALLBACK_BBOX = [-9.5, 35.9, 4.6, 43.8]  # [min_lon, min_lat, max_lon, max_lat]
FALLBACK_GEOJSON_PATH = None

# Google Earth Engine (required: create project at https://console.cloud.google.com/earth-engine)
GEE_PROJECT = "ie-microsoft-capstone"  # your Cloud project ID

# S3 (same pattern as gbif notebooks)
S3_BUCKET = "ie-datalake"
BRONZE_PREFIX = "bronze/gee_hex_terrain"  # raw GEE output
GOLD_PREFIX = "gold/gee_hex_terrain"  # aggregated stats: country/snapshot/h3_resolution
AWS_PROFILE = "486717354268_PowerUserAccess"
PARQUET_COMPRESSION = "snappy"

# GEE Export (jednorazowo) – rastry na S3
RUN_GEE_EXPORT = True  # False = skip, używaj już zapisanych rastrów na S3

# Gdzie eksportować: "local" (bezpośrednio na dysk) | "gcs" | "drive"
EXPORT_DESTINATION = "local"  # local = pobierz do data/ bez Drive/GCS

GCS_BUCKET = "ie-microsoft-capstone-gee"  # tylko gdy EXPORT_DESTINATION="gcs"
# src/data/gee_rasters – rastry landcover i elevation
LOCAL_RASTER_DIR = REPO_ROOT / "src" / "data" / "gee_rasters"
LOCAL_GRID_N = 6                         # local: siatka NxN kafelków (każdy <32MB)

# Rastry na S3 (s3://S3_BUCKET/S3_RASTER_PREFIX/)
S3_RASTER_PREFIX = "bronze/gee_rasters"
LANDCOVER_TIF = "landcover_spain.tif"
ELEVATION_TIF = "elevation_spain.tif"
SPAIN_GEOJSON = "spain_boundary.geojson"

In [ ]:
%pip install -q earthengine-api h3 pandas pyarrow s3fs rasterio rasterstats geopandas google-cloud-storage requests

In [3]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import ee
import h3
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("gee_hex")

fs = s3fs.S3FileSystem(profile=AWS_PROFILE)
s3_raster_prefix = f"{S3_BUCKET}/{S3_RASTER_PREFIX}"

if RUN_GEE_EXPORT:
    try:
        ee.Initialize(project=GEE_PROJECT)
        log.info("Earth Engine initialized (project=%s)", GEE_PROJECT)
    except Exception as e:
        log.error("Run: earthengine authenticate\n%s", e)
        raise

18:00:26 [INFO] Earth Engine initialized (project=ie-microsoft-capstone)


## 1) Spain geometry

In [4]:
def get_spain_geometry() -> ee.Geometry:
    """Spain boundary from FAO GAUL level0."""
    countries = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
    spain = countries.filter(ee.Filter.eq("ADM0_NAME", "Spain")).first()
    return spain.geometry()


def bbox_to_geometry(bbox: list[float]) -> ee.Geometry:
    """[min_lon, min_lat, max_lon, max_lat] -> ee.Geometry.Rectangle."""
    return ee.Geometry.Rectangle(bbox)


def geojson_to_geometry(path: str) -> ee.Geometry:
    """Load GeoJSON file and convert to ee.Geometry."""
    with open(path) as f:
        gj = json.load(f)
    return ee.Geometry(gj)


spain_geojson_s3 = f"s3://{S3_BUCKET}/{S3_RASTER_PREFIX}/{SPAIN_GEOJSON}"

if RUN_GEE_EXPORT:
    if USE_SPAIN_BOUNDARY:
        region = get_spain_geometry()
        log.info("Using Spain boundary from FAO GAUL")
    elif FALLBACK_GEOJSON_PATH:
        region = geojson_to_geometry(FALLBACK_GEOJSON_PATH)
        log.info("Using geometry from %s", FALLBACK_GEOJSON_PATH)
    else:
        region = bbox_to_geometry(FALLBACK_BBOX)
        log.info("Using fallback bbox %s", FALLBACK_BBOX)
    # Save boundary to S3 for local processing (no GEE needed later)
    gj = region.getInfo()
    with fs.open(spain_geojson_s3, "w") as f:
        json.dump(gj, f, indent=2)
    log.info("Saved boundary to %s", spain_geojson_s3)
else:
    with fs.open(spain_geojson_s3, "r") as f:
        region_geojson = json.load(f)
    region = region_geojson  # dla lokalnego H3; GEE nie jest potrzebne
    log.info("Loaded boundary from %s", spain_geojson_s3)

18:00:29 [INFO] Using Spain boundary from FAO GAUL
18:00:30 [INFO] Found credentials in shared credentials file: ~/.aws/credentials
18:00:31 [INFO] Saved boundary to s3://ie-datalake/bronze/gee_rasters/spain_boundary.geojson


## 2) H3 hex grid

In [5]:
def _extract_polygon_rings(gj: dict) -> list:
    """Extract exterior rings from GeoJSON geometry (Polygon, MultiPolygon, GeometryCollection)."""
    geotype = gj.get("type", "")
    if geotype == "Polygon":
        coords = gj.get("coordinates", [])
        return [coords[0]] if coords else []
    if geotype == "MultiPolygon":
        coords = gj.get("coordinates", [])
        return [p[0] for p in coords] if coords else []
    if geotype == "GeometryCollection":
        rings = []
        for g in gj.get("geometries", []):
            rings.extend(_extract_polygon_rings(g))
        return rings
    return []


def get_h3_hexes_for_region(region_ee_or_geojson, resolution: int) -> list[str]:
    """
    Get H3 hex IDs covering the region.
    region_ee_or_geojson: ee.Geometry (calls getInfo) or GeoJSON dict.
    Handles Polygon, MultiPolygon, GeometryCollection (e.g. FAO GAUL Spain).
    """
    gj = region_ee_or_geojson.getInfo() if hasattr(region_ee_or_geojson, "getInfo") else region_ee_or_geojson
    if not gj:
        raise ValueError("Empty geometry")

    polygons = _extract_polygon_rings(gj)
    if not polygons:
        raise ValueError(f"No polygons in geometry type: {gj.get('type', '')}")

    hexes = set()
    for ring in polygons:
        if len(ring) < 3:
            continue
        if ring[0] != ring[-1]:
            ring = list(ring) + [ring[0]]
        geojson = {"type": "Polygon", "coordinates": [ring]}
        try:
            cells = h3.geo_to_cells(geojson, resolution)
            hexes.update(cells)
        except Exception as e:
            log.warning("Polyfill failed: %s", e)
    return sorted(hexes)


def hex_to_ee_feature(h3_index: str, resolution: int) -> dict:
    """Convert H3 index to GeoJSON-like feature for ee.FeatureCollection."""
    # h3 v4: cell_to_boundary returns (lat, lng); GeoJSON needs [lng, lat]
    boundary = h3.cell_to_boundary(h3_index)
    boundary = [[p[1], p[0]] for p in boundary]  # (lat,lng) -> [lng,lat]
    if boundary[0] != boundary[-1]:
        boundary = list(boundary) + [boundary[0]]
    return {
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [boundary]},
        "properties": {"h3_index": h3_index, "h3_resolution": resolution},
    }


# Copernicus discrete_classification class IDs (Level 1 simplified)
LC_CLASS_IDS = [0, 20, 30, 40, 50, 60, 70, 80, 90, 100, 111, 112, 113, 114, 115, 116, 121, 122, 123, 124, 125, 126, 200]
LC_CLASS_NAMES = {
    0: "unknown", 20: "shrubs", 30: "herbaceous", 40: "crops", 50: "urban",
    60: "bare", 70: "snow_ice", 80: "water_permanent", 90: "wetland", 100: "moss",
    111: "forest_evergreen_needle", 112: "forest_evergreen_broad", 113: "forest_deciduous_needle",
    114: "forest_deciduous_broad", 115: "forest_mixed", 116: "forest_other",
    121: "open_forest_needle", 122: "open_forest_broad", 123: "open_forest_deciduous_needle",
    124: "open_forest_deciduous_broad", 125: "open_forest_mixed", 126: "open_forest_other",
    200: "ocean",
}

## 3) GEE data & aggregation

In [6]:
def load_landcover(year: int) -> ee.Image:
    """Copernicus Land Cover 100m, discrete_classification band."""
    return ee.Image(f"COPERNICUS/Landcover/100m/Proba-V-C3/Global/{year}").select("discrete_classification")


def load_elevation() -> ee.Image:
    """SRTM elevation + optional slope. Cast to Float so bands match (SRTM=Int16, slope=Float32)."""
    dem = ee.Image("USGS/SRTMGL1_003").select("elevation").toFloat()
    if INCLUDE_SLOPE:
        slope = ee.Terrain.slope(dem)
        return dem.addBands(slope)
    return dem


def build_hex_fc(hex_ids: list[str], resolution: int) -> ee.FeatureCollection:
    """Build ee.FeatureCollection from H3 hex IDs."""
    features = [hex_to_ee_feature(h, resolution) for h in hex_ids]
    return ee.FeatureCollection(features)


def add_landcover_proportions_simple(fc: ee.FeatureCollection, lc_image: ee.Image) -> ee.FeatureCollection:
    """reduceRegions with frequencyHistogram. Proportions computed in Python after getInfo."""
    return lc_image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.frequencyHistogram(),
        scale=LANDCOVER_SCALE,
    )


def add_elevation_stats(fc: ee.FeatureCollection, elev_image: ee.Image) -> ee.FeatureCollection:
    """Add elevation_mean and slope_mean per hex."""
    return elev_image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=ELEVATION_SCALE,
    )


def write_to_s3(df_bronze: pd.DataFrame, df_gold: pd.DataFrame, country: str, snapshot: str, h3_resolution: int) -> None:
    """Write bronze (raw) and gold (aggregated stats) to S3. Partition: country/snapshot/h3_resolution."""
    base_bronze = f"{S3_BUCKET}/{BRONZE_PREFIX}/country={country}/snapshot={snapshot}/h3_resolution={h3_resolution}"
    base_gold = f"{S3_BUCKET}/{GOLD_PREFIX}/country={country}/snapshot={snapshot}/h3_resolution={h3_resolution}"
    table_bronze = pa.Table.from_pandas(df_bronze, preserve_index=False)
    table_gold = pa.Table.from_pandas(df_gold, preserve_index=False)
    pq.write_to_dataset(table_bronze, root_path=f"s3://{base_bronze}", filesystem=fs, existing_data_behavior="delete_matching", compression=PARQUET_COMPRESSION)
    pq.write_to_dataset(table_gold, root_path=f"s3://{base_gold}", filesystem=fs, existing_data_behavior="delete_matching", compression=PARQUET_COMPRESSION)
    log.info("  Bronze: s3://%s", base_bronze)
    log.info("  Gold:   s3://%s (%d rows)", base_gold, len(df_gold))

In [14]:
## 3a) GEE Export – rastry na S3 (GEE→GCS/Drive→S3)

if RUN_GEE_EXPORT:
    import time

    lc_image = load_landcover(LANDCOVER_YEAR)
    elev_image = load_elevation()
    prefix = f"gee_rasters_{COUNTRY}_{SNAPSHOT}"
    drive_folder = "GEE_Exports"

    if EXPORT_DESTINATION == "gcs":
        from google.cloud import storage

        task_lc = ee.batch.Export.image.toCloudStorage(
            image=lc_image.clip(region),
            description=f"landcover_{COUNTRY}_{SNAPSHOT}",
            bucket=GCS_BUCKET,
            fileNamePrefix=f"{prefix}/landcover_spain",
            region=region,
            scale=LANDCOVER_SCALE,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
        )
        task_lc.start()
        log.info("Export land cover: %s", task_lc.id)
        task_el = ee.batch.Export.image.toCloudStorage(
            image=elev_image.clip(region),
            description=f"elevation_{COUNTRY}_{SNAPSHOT}",
            bucket=GCS_BUCKET,
            fileNamePrefix=f"{prefix}/elevation_spain",
            region=region,
            scale=ELEVATION_SCALE,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
        )
        task_el.start()
        log.info("Export elevation+slope: %s", task_el.id)

        for task, name in [(task_lc, "landcover"), (task_el, "elevation")]:
            while task.active():
                log.info("  %s: czekam...", name)
                time.sleep(30)
            status = task.status()
            if status.get("state") == "COMPLETED":
                gcs_blob = f"{prefix}/{name}_spain.tif"
                gcs_client = storage.Client(project=GEE_PROJECT)
                bucket = gcs_client.bucket(GCS_BUCKET)
                blob = bucket.blob(gcs_blob)
                data = blob.download_as_bytes()
                s3_key = f"{S3_RASTER_PREFIX}/{name}_spain.tif"
                with fs.open(f"s3://{S3_BUCKET}/{s3_key}", "wb") as f:
                    f.write(data)
                log.info("  %s → s3://%s/%s", name, S3_BUCKET, s3_key)
            else:
                log.error("  %s failed: %s", name, status.get("error_message", status))

    elif EXPORT_DESTINATION == "local":  # pobierz bezpośrednio na dysk (kafelki)
        import io
        import zipfile
        import requests
        import rasterio
        from rasterio.merge import merge as rasterio_merge
        from shapely.geometry import shape

        out_dir = Path(LOCAL_RASTER_DIR)
        out_dir.mkdir(parents=True, exist_ok=True)
        tiles_dir = out_dir / "tiles"
        tiles_dir.mkdir(exist_ok=True)

        shp = shape(region.getInfo())
        minx, miny, maxx, maxy = shp.bounds
        n = LOCAL_GRID_N
        dx, dy = (maxx - minx) / n, (maxy - miny) / n

        def download_and_merge(name: str, image: ee.Image, scale: int, out_path: Path):
            srcs = []
            for i in range(n):
                for j in range(n):
                    x1, y1 = minx + i * dx, miny + j * dy
                    x2, y2 = minx + (i + 1) * dx, miny + (j + 1) * dy
                    rect = ee.Geometry.Rectangle([x1, y1, x2, y2])
                    clip = image.clip(rect)
                    url = clip.getDownloadURL({"region": rect, "scale": scale, "format": "ZIPPED_GEO_TIFF", "filePerBand": False})
                    tile_path = tiles_dir / f"{name}_{i}_{j}.tif"
                    r = requests.get(url, timeout=120)
                    r.raise_for_status()
                    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                        for fn in z.namelist():
                            if fn.endswith(".tif"):
                                with open(tile_path, "wb") as tf:
                                    tf.write(z.read(fn))
                                break
                    srcs.append(rasterio.open(tile_path))
                    log.info("  %s tile %d/%d", name, i * n + j + 1, n * n)
            merged, out_transform = rasterio_merge(srcs)
            meta = srcs[0].meta.copy()
            for s in srcs:
                s.close()
            meta.update(width=merged.shape[2], height=merged.shape[1], transform=out_transform)
            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(merged)
            log.info("  %s → %s", name, out_path)

        lc_path = out_dir / LANDCOVER_TIF
        elev_path = out_dir / ELEVATION_TIF
        download_and_merge("landcover", lc_image.clip(region), LANDCOVER_SCALE, lc_path)
        download_and_merge("elevation", elev_image.clip(region), ELEVATION_SCALE, elev_path)

        for name, p in [("landcover", lc_path), ("elevation", elev_path)]:
            s3_key = f"{S3_RASTER_PREFIX}/{p.name}"
            with open(p, "rb") as f:
                data = f.read()
            with fs.open(f"s3://{S3_BUCKET}/{s3_key}", "wb") as f:
                f.write(data)
            log.info("  %s → s3://%s/%s", name, S3_BUCKET, s3_key)

    else:  # drive: GEE→Drive, potem pobierz lokalnie i uruchom komórkę 3b
        task_lc = ee.batch.Export.image.toDrive(
            image=lc_image.clip(region),
            description=f"landcover_{COUNTRY}_{SNAPSHOT}",
            folder=drive_folder,
            region=region,
            scale=LANDCOVER_SCALE,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
            fileNamePrefix="landcover_spain",
        )
        task_lc.start()
        task_el = ee.batch.Export.image.toDrive(
            image=elev_image.clip(region),
            description=f"elevation_{COUNTRY}_{SNAPSHOT}",
            folder=drive_folder,
            region=region,
            scale=ELEVATION_SCALE,
            maxPixels=1e13,
            fileFormat="GeoTIFF",
            fileNamePrefix="elevation_spain",
        )
        task_el.start()
        log.info("Export land cover: %s", task_lc.id)
        log.info("Export elevation+slope: %s", task_el.id)

        for task, name in [(task_lc, "landcover"), (task_el, "elevation")]:
            while task.active():
                log.info("  %s: czekam...", name)
                time.sleep(30)
            status = task.status()
            if status.get("state") == "COMPLETED":
                log.info("  %s: gotowe na Drive (folder: %s)", name, drive_folder)
            else:
                log.error("  %s failed: %s", name, status.get("error_message", status))

        log.info("")
        log.info("Pobierz pliki z drive.google.com (folder '%s'): landcover_spain.tif, elevation_spain.tif", drive_folder)
        log.info("Zapisz je w: %s", LOCAL_RASTER_DIR)
        log.info("Następnie uruchom komórkę 3b (Upload z lokalnego do S3)")

17:39:30 [INFO]   landcover tile 1/36


KeyboardInterrupt: 

In [9]:
## 3b) Upload z lokalnego do S3 (gdy używasz Drive)

# Uruchom po pobraniu plików z Drive do LOCAL_RASTER_DIR
if RUN_GEE_EXPORT and EXPORT_DESTINATION == "drive":
    local_dir = Path(LOCAL_RASTER_DIR)
    lc_local = local_dir / LANDCOVER_TIF
    elev_local = local_dir / ELEVATION_TIF
    if lc_local.exists() and elev_local.exists():
        for name, p in [("landcover", lc_local), ("elevation", elev_local)]:
            s3_key = f"{S3_RASTER_PREFIX}/{p.name}"
            with open(p, "rb") as f:
                data = f.read()
            with fs.open(f"s3://{S3_BUCKET}/{s3_key}", "wb") as f:
                f.write(data)
            log.info("%s → s3://%s/%s", name, S3_BUCKET, s3_key)
    else:
        log.warning("Brak plików w %s. Pobierz z Drive: %s, %s", local_dir, LANDCOVER_TIF, ELEVATION_TIF)

## 4) Lokalne przetwarzanie – rasterstats → S3

### Diagnostyka rastrów (opcjonalnie)

Uruchom poniższą komórkę **przed** przetwarzaniem, żeby sprawdzić czy TIFy są poprawne. Pokaże: wartości w rastrze, overlap z Hiszpanią, test zonal_stats na 1 heksie.

In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# PARAMETRY PRZETWARZANIA (rasterstats → S3)
# ═══════════════════════════════════════════════════════════════════════════════
# Uwaga: RUN_GEE_EXPORT (sekcja 3a) eksportuje OBA rastry. Tu decydujesz, które UŻYĆ.
# PROCESS_LANDCOVER=False → lc_*_pct będą zerami! Oba=True daje elevation, slope i procenty.
PROCESS_LANDCOVER = True
PROCESS_ELEVATION = True
# Resolutions do przetworzenia (9 pomijamy – za wolne; można dodać później)
PROCESS_RESOLUTIONS = [6, 7, 8]
# Wznowienie: (res, hex_start) np. (8, 100000); None = od początku
RESUME_FROM = None

import tempfile
import numpy as np
import geopandas as gpd
import pyarrow.parquet as pq
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import Polygon

log.info("=== Lokalne przetwarzanie (rasterstats → S3) ===")
log.info("PROCESS_LANDCOVER=%s, PROCESS_ELEVATION=%s", PROCESS_LANDCOVER, PROCESS_ELEVATION)

need_lc = PROCESS_LANDCOVER
need_elev = PROCESS_ELEVATION
resume = RESUME_FROM  # (res, hex_start) lub None

# LOCAL_RASTER_DIR jest już względem REPO_ROOT (działa z src/ i src/gee/)
local_dir = Path(LOCAL_RASTER_DIR).resolve()
lc_local = local_dir / LANDCOVER_TIF
elev_local = local_dir / ELEVATION_TIF
lc_s3 = f"{S3_BUCKET}/{S3_RASTER_PREFIX}/{LANDCOVER_TIF}"
elev_s3 = f"{S3_BUCKET}/{S3_RASTER_PREFIX}/{ELEVATION_TIF}"

log.info("Szukam rastrów: local_dir=%s", local_dir)
log.info("  landcover: %s (exists=%s)", lc_local, lc_local.exists())
log.info("  elevation: %s (exists=%s)", elev_local, elev_local.exists())

tmp = None
lc_path = elev_path = None
if need_lc:
    if lc_local.exists():
        lc_path = lc_local
        log.info("✓ landcover: używam lokalnego %s", lc_path)
    elif fs.exists(lc_s3):
        log.info("Pobieram landcover z S3 (może potrwać)...")
        tmp = tempfile.mkdtemp() if tmp is None else tmp
        lc_path = Path(tmp) / LANDCOVER_TIF
        fs.get(lc_s3, str(lc_path))
        log.info("✓ landcover: pobrano z S3 → %s", lc_path)
    else:
        raise FileNotFoundError(f"Brak landcover w {local_dir} ani s3://{lc_s3}. Uruchom najpierw GEE Export (sekcja 3a) z RUN_GEE_EXPORT=True.")
if need_elev:
    if elev_local.exists():
        elev_path = elev_local
        log.info("✓ elevation: używam lokalnego %s", elev_path)
    elif fs.exists(elev_s3):
        log.info("Pobieram elevation z S3 (może potrwać)...")
        tmp = tempfile.mkdtemp() if tmp is None else tmp
        elev_path = Path(tmp) / ELEVATION_TIF
        fs.get(elev_s3, str(elev_path))
        log.info("✓ elevation: pobrano z S3 → %s", elev_path)
    else:
        raise FileNotFoundError(f"Brak elevation w {local_dir} ani s3://{elev_s3}. Uruchom najpierw GEE Export (sekcja 3a) z RUN_GEE_EXPORT=True.")
if not need_lc and not need_elev:
    raise ValueError("Ustaw PROCESS_LANDCOVER=True lub PROCESS_ELEVATION=True")

log.info("Rastry gotowe. Rozpoczynam zonal_stats dla resolutions %s (bez 9 – za wolne)...", PROCESS_RESOLUTIONS)
if resume:
    log.info("Wznowienie: res=%d od hexa %d", resume[0], resume[1])

try:
    all_dfs = []
    for res in PROCESS_RESOLUTIONS:
        if resume and res < resume[0]:
            log.info("Resolution %d: pomijam (wznowienie od res %d)", res, resume[0])
            continue
        log.info("Resolution %d: generating hex grid...", res)
        hex_ids = get_h3_hexes_for_region(region, res)
        n_hex = len(hex_ids)
        if resume and res == resume[0]:
            hex_start = min(resume[1], n_hex)
            hex_ids = hex_ids[hex_start:]
            log.info("  %d hexes (wznowienie od indeksu %d, zostaje %d)", n_hex, hex_start, len(hex_ids))
        else:
            log.info("  %d hexes", n_hex)
        if n_hex == 0 or len(hex_ids) == 0:
            continue

        # GeoDataFrame hexów
        geoms = []
        for h in hex_ids:
            boundary = h3.cell_to_boundary(h)
            coords = [[p[1], p[0]] for p in boundary]  # (lat,lng) -> [lng,lat]
            if coords[0] != coords[-1]:
                coords.append(coords[0])
            geoms.append(Polygon(coords))
        gdf = gpd.GeoDataFrame({"h3_index": hex_ids, "h3_resolution": res}, geometry=geoms, crs="EPSG:4326")
        n_chunk = len(hex_ids)

        # Landcover: EPSG:3857 – reprojektuj geometrię. Elevation: EPSG:4326 – gdf zostaje w 4326.
        gdf_lc = gdf.to_crs("EPSG:3857") if need_lc else gdf

        lc_pct = {f"lc_{cid}_pct": [] for cid in LC_CLASS_IDS}
        elev_means = [None] * n_chunk
        slope_means = [None] * n_chunk if INCLUDE_SLOPE else []

        if need_lc:
            log.info("  res=%d: landcover via rasterio.mask (%d hexów)...", res, n_chunk)
            from rasterio.mask import mask as rasterio_mask
            from collections import Counter
            stats_lc = []
            with rasterio.open(str(lc_path)) as src:
                for idx, geom in enumerate(gdf_lc.geometry):
                    try:
                        out_data, _ = rasterio_mask(src, [geom], crop=True, all_touched=True, filled=False)
                        valid = out_data[0].compressed().astype(int)
                        hist = dict(Counter(valid)) if len(valid) > 0 else {}
                    except Exception:
                        hist = {}
                    stats_lc.append({"histogram": hist})
                    if (idx + 1) % 2000 == 0:
                        log.info("    landcover %d/%d", idx + 1, n_chunk)
            for s in stats_lc:
                raw = s.get("histogram", {}) or {}
                hist = {int(k): v for k, v in raw.items()}
                total = sum(hist.values()) or 1
                for cid in LC_CLASS_IDS:
                    lc_pct[f"lc_{cid}_pct"].append(hist.get(cid, 0) / total)
            any_nonzero = any(v > 0 for col in lc_pct for v in lc_pct[col])
            if not any_nonzero:
                raise ValueError(
                    "Landcover: wszystkie lc_*_pct są zerami! Sprawdź raster %s i CRS." % lc_path
                )
            log.info("  ✓ landcover OK")
        else:
            stats_lc = [{"histogram": {}}] * n_chunk
            for cid in LC_CLASS_IDS:
                lc_pct[f"lc_{cid}_pct"] = [0.0] * n_chunk  # brak landcover – wypełnij zerami

        if need_elev:
            log.info("  res=%d: zonal_stats elevation+slope...", res)
            with rasterio.open(elev_path) as src:
                stats_el = zonal_stats(gdf, src.read(1), affine=src.transform, stats="mean", nodata=src.nodata)
                stats_sl = zonal_stats(gdf, src.read(2), affine=src.transform, stats="mean", nodata=src.nodata)
            elev_means = [s.get("mean") for s in stats_el]
            slope_means = [s.get("mean") for s in stats_sl] if INCLUDE_SLOPE else [None] * n_chunk

        # Gold
        gold_rows = [{"h3_index": h, "h3_resolution": res, "elevation_mean": e, "slope_mean": s} for h, e, s in zip(hex_ids, elev_means, slope_means)]
        for k, v in lc_pct.items():
            for i, row in enumerate(gold_rows):
                row[k] = v[i]
        df_gold = pd.DataFrame(gold_rows)
        # Sanityzacja: -inf/np.inf -> np.nan (hexy bez pikseli lub nodata)
        for col in ["elevation_mean", "slope_mean"]:
            if col in df_gold.columns:
                df_gold[col] = df_gold[col].replace([np.inf, -np.inf], np.nan)

        # Bronze
        bronze_rows = []
        for i, h in enumerate(hex_ids):
            hist = {int(k): v for k, v in (stats_lc[i].get("histogram") or {}).items()}
            bronze_rows.append({
                "h3_index": h, "h3_resolution": res,
                "elevation": elev_means[i], "slope": slope_means[i] if INCLUDE_SLOPE else None,
                "discrete_classification": json.dumps(hist),
            })
        df_bronze = pd.DataFrame(bronze_rows)

        # Przy wznowieniu: dołącz do istniejącej partycji na S3
        if resume and res == resume[0] and resume[1] > 0:
            base_gold = f"{S3_BUCKET}/{GOLD_PREFIX}/country={COUNTRY}/snapshot={SNAPSHOT}/h3_resolution={res}"
            base_bronze = f"{S3_BUCKET}/{BRONZE_PREFIX}/country={COUNTRY}/snapshot={SNAPSHOT}/h3_resolution={res}"
            try:
                existing_gold = pq.read_table(f"s3://{base_gold}", filesystem=fs)
                existing_bronze = pq.read_table(f"s3://{base_bronze}", filesystem=fs)
                df_ex_g = existing_gold.to_pandas()
                df_ex_b = existing_bronze.to_pandas()
                df_ex_g = df_ex_g.iloc[: resume[1]]
                df_ex_b = df_ex_b.iloc[: resume[1]]
                df_gold = pd.concat([df_ex_g, df_gold], ignore_index=True)
                df_bronze = pd.concat([df_ex_b, df_bronze], ignore_index=True)
                log.info("  Scalono z istniejącymi %d wierszami", resume[1])
            except Exception as e:
                log.warning("  Brak istniejącej partycji lub błąd odczytu: %s. Zapisuję tylko nowe.", e)

        all_dfs.append(df_gold)
        write_to_s3(df_bronze, df_gold, COUNTRY, SNAPSHOT, res)
finally:
    if tmp:
        import shutil
        shutil.rmtree(tmp, ignore_errors=True)

18:09:55 [INFO] === Lokalne przetwarzanie (rasterstats → S3) ===
18:09:55 [INFO] PROCESS_LANDCOVER=True, PROCESS_ELEVATION=True
18:09:55 [INFO] Szukam rastrów: local_dir=/Users/jakubizewski/Desktop/repos/ie-microsoft-capstone/src/data/gee_rasters
18:09:55 [INFO]   landcover: /Users/jakubizewski/Desktop/repos/ie-microsoft-capstone/src/data/gee_rasters/landcover_spain.tif (exists=True)
18:09:55 [INFO]   elevation: /Users/jakubizewski/Desktop/repos/ie-microsoft-capstone/src/data/gee_rasters/elevation_spain.tif (exists=True)
18:09:55 [INFO] ✓ landcover: używam lokalnego /Users/jakubizewski/Desktop/repos/ie-microsoft-capstone/src/data/gee_rasters/landcover_spain.tif
18:09:55 [INFO] ✓ elevation: używam lokalnego /Users/jakubizewski/Desktop/repos/ie-microsoft-capstone/src/data/gee_rasters/elevation_spain.tif
18:09:55 [INFO] Rastry gotowe. Rozpoczynam zonal_stats dla resolutions [6, 7, 8] (bez 9 – za wolne)...
18:09:55 [INFO] Resolution 6: generating hex grid...
18:09:57 [INFO]   13641 hexes
1

In [7]:
# Diagnostyka rastrów – lekka wersja (bez ładowania całego rastra do RAM)
import rasterio
from pathlib import Path

lc_path = Path(LOCAL_RASTER_DIR) / LANDCOVER_TIF
elev_path = Path(LOCAL_RASTER_DIR) / ELEVATION_TIF

print("=== Landcover raster (metadata tylko) ===")
if lc_path.exists():
    with rasterio.open(str(lc_path)) as src:
        print(f"  Ścieżka: {lc_path}")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
        print(f"  Shape: {src.width} x {src.height}, dtype: {src.dtypes[0]}")
        print(f"  nodata: {src.nodata}")
        # Próbka 100x100 pikseli ze środka (bez ładowania całości)
        h, w = min(100, src.height), min(100, src.width)
        row_off, col_off = (src.height - h) // 2, (src.width - w) // 2
        arr = src.read(1, window=((row_off, row_off + h), (col_off, col_off + w)))
        nodata = src.nodata if src.nodata is not None else -9999
        valid = arr[arr != nodata]
        if len(valid) > 0:
            uniq = set(int(v) for v in valid.flat)
            print(f"  Próbka środka: unikalne wartości = {sorted(uniq)[:15]}...")
        else:
            print("  Próbka: same nodata – możliwy pusty raster")
else:
    print(f"  BRAK: {lc_path}")

print("\n=== Elevation raster ===")
if elev_path.exists():
    with rasterio.open(str(elev_path)) as src:
        print(f"  Bands: {src.count}, CRS: {src.crs}, Bounds: {src.bounds}")

=== Landcover raster (metadata tylko) ===
  Ścieżka: /Users/jakubizewski/Desktop/repos/ie-microsoft-capstone/src/data/gee_rasters/landcover_spain.tif
  CRS: EPSG:3857
  Bounds: BoundingBox(left=-2022750.0, bottom=3203400.0, right=480650.0, top=5435800.0)
  Shape: 12517 x 11162, dtype: uint8
  nodata: 0.0
  Próbka środka: unikalne wartości = [200]...

=== Elevation raster ===
  Bands: 2, CRS: EPSG:4326, Bounds: BoundingBox(left=-18.170437085325347, bottom=27.637432463199012, right=4.317987737322753, top=43.80890420791864)


In [ ]:
if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    log.info("Total: %d rows across %d resolutions", len(df_all), len(all_dfs))
    display(df_all.head(10))

In [ ]:
# (Komórkę powyżej możesz usunąć – duplikat wyświetlania)

## Walidacja Gold Layer

Preview + sprawdzenie czy elevation i land cover zapisały się poprawnie.

In [12]:
# Walidacja gold layer – uruchom po zapisie do S3
# Uwaga: s3fs zwraca ścieżki bez s3:// – trzeba użyć bucket/path, nie s3://bucket/path
# (inaczej PyArrow: "GetFileInfo yielded path ... outside base dir")
import pyarrow.dataset as ds

VALIDATE_RES = 7  # resolution do walidacji (6/7/8/9)
gold_prefix = f"{S3_BUCKET}/{GOLD_PREFIX}/country={COUNTRY}/snapshot={SNAPSHOT}/h3_resolution={VALIDATE_RES}"
files = fs.glob(f"{gold_prefix}/*.parquet")

try:
    if not files:
        raise FileNotFoundError(f"Brak plików parquet w {gold_prefix}")
    tbl = ds.dataset(files, filesystem=fs, format="parquet").scanner().to_table()
    df = tbl.to_pandas()
except Exception as e:
    print(f"❌ Nie można załadować gold: {e}")
    raise

df
# print(f"✓ Załadowano {len(df):,} wierszy (res {VALIDATE_RES})")
# print(f"  Kolumny: {list(df.columns)}")

# # 1) Elevation + slope
# elev_ok = "elevation_mean" in df.columns
# slope_ok = "slope_mean" in df.columns
# print(f"\n📐 Elevation: {'✓' if elev_ok else '✗'} elevation_mean")
# print(f"📐 Slope:     {'✓' if slope_ok else '✗'} slope_mean")

# if elev_ok:
#     n_null = df["elevation_mean"].isna().sum()
#     rng = (df["elevation_mean"].min(), df["elevation_mean"].max())
#     print(f"   zakres: {rng[0]:.0f}–{rng[1]:.0f} m, null: {n_null}")
# if slope_ok:
#     n_null = df["slope_mean"].isna().sum()
#     rng = (df["slope_mean"].min(), df["slope_mean"].max())
#     print(f"   zakres: {rng[0]:.1f}–{rng[1]:.1f}°, null: {n_null}")

# # 2) Land cover
# lc_cols = [c for c in df.columns if c.startswith("lc_") and c.endswith("_pct")]
# print(f"\n🗺️ Land cover: {'✓' if lc_cols else '✗'} {len(lc_cols)} klas (lc_*_pct)")

# if lc_cols:
#     lc_sum = df[lc_cols].sum(axis=1)
#     ok_sum = (lc_sum > 0.99) & (lc_sum < 1.01)
#     n_bad = (~ok_sum).sum()
#     print(f"   suma % na wiersz: min={lc_sum.min():.3f}, max={lc_sum.max():.3f}")
#     if n_bad > 0:
#         print(f"   ⚠️ {n_bad} wierszy ma sumę ≠ 1.0")
#     else:
#         print(f"   ✓ wszystkie wiersze: suma ≈ 1.0")

# # 3) Preview
# print("\n📋 Preview (5 losowych wierszy):")
# sample = df.sample(n=min(5, len(df)), random_state=42)
# preview_cols = ["h3_index"]
# if "elevation_mean" in df.columns:
#     preview_cols.append("elevation_mean")
# if "slope_mean" in df.columns:
#     preview_cols.append("slope_mean")
# preview_cols += (lc_cols[:5] if lc_cols else [])
# display(sample[[c for c in preview_cols if c in sample.columns]])

,h3_index,h3_resolution,elevation_mean,slope_mean,lc_0_pct,lc_20_pct,lc_30_pct,lc_40_pct,lc_50_pct,lc_60_pct,...,lc_114_pct,lc_115_pct,lc_116_pct,lc_121_pct,lc_122_pct,lc_123_pct,lc_124_pct,lc_125_pct,lc_126_pct,lc_200_pct
0,871849098ffffff,7,NaN,18.174881,0.0,0.010753,0.232975,0.00000,0.000000,0.000000,...,0.379928,0.0,0.039427,0.0,0.0,0.0,0.010753,0.0,0.326165,0.000000
1,871849099ffffff,7,NaN,20.108670,0.0,0.003610,0.021661,0.00000,0.000000,0.000000,...,0.888087,0.0,0.025271,0.0,0.0,0.0,0.021661,0.0,0.039711,0.000000
2,87184909affffff,7,519.645714,16.682818,0.0,0.003597,0.068345,0.00000,0.000000,0.000000,...,0.665468,0.0,0.071942,0.0,0.0,0.0,0.010791,0.0,0.179856,0.000000
3,87184909bffffff,7,581.821839,24.036784,0.0,0.014337,0.071685,0.00000,0.000000,0.000000,...,0.842294,0.0,0.025090,0.0,0.0,0.0,0.010753,0.0,0.035842,0.000000
4,8718490dbffffff,7,NaN,11.573736,0.0,0.000000,0.285714,0.00000,0.000000,0.000000,...,0.564286,0.0,0.050000,0.0,0.0,0.0,0.010714,0.0,0.089286,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95476,875536da6ffffff,7,75.751479,8.388490,0.0,0.013575,0.072398,0.00000,0.072398,0.579186,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.262443
95477,875536db0ffffff,7,NaN,13.471022,0.0,0.000000,0.017937,0.00000,0.143498,0.690583,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.147982
95478,875536db4ffffff,7,350.621302,21.250618,0.0,0.153153,0.675676,0.00000,0.000000,0.171171,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
95479,875536db5ffffff,7,160.207317,13.096275,0.0,0.053571,0.133929,0.03125,0.031250,0.745536,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.004464,0.000000
